# Теория


Linux - ядро операционной системы, программа которая управляет процессором, памятью, устройсвами и запущенными процессорами.
Когда говорят про Linux обычно имею ввиду ядро Linux + утилиты GNU + другие программы.

GNU - расшифровывается буквально как "GNU's Not Unix!", где буква G как раз таки и обозначает GNU, образуя бесконечную рекурсию, повторяющую, что это не Unix.

### Запуск Linux
#### 1) BIOS

Запуск начинается с прошивки материнской платы BIOS/UEFI. Она запускает P.O.S.T. - Power-on self-test , проверяюий наличие памяти, работу видеокарты, инициализировался ли процессор, нашёл ли он оперативку. После чего ищет, с чего бы можно загрузится.
#### 2) MBT

После того, как был найден жёсткий диск, BIOS считывает первые 512 байт диска MBT - Master Boot Record, 

**Master boot record**
|Код загрузчика|Таблица разделов|Сигнатура|
|-|-|-|
|446 байт|64 байта (4 раздела по 16 байт)|2 байта|

**Сигнатура**

55AA - в противном случае BIOS не считает этот сектор загрузочным 


**Раздел** в *таблице разделов* имеет следующую структуру:

|Флаг активности|CHS-aдрес начала раздела|Тип раздела|CHS-адрес конца раздела|LBA-адрес начала|Размер раздела в секторах|
|-|-|-|-|-|-|
|1 байт|3 байта|1 байт|3 байта|4 байта|4 байта|

Ранее испольховался CHS-адрес, но с добавлением MBR решили использовать LBA, при этом сохранив CHS, чтобы не ломать совместимость.

**Код загрузчика** 

Это программа перебирающая 4 раздела и ищущая среди них флаг 0x80 

|Код загрузчика|Сигнатура|Код загрузчика|Таблица разделов|VBR|
|-|-|-|-|-|
|Загрузка кода в память|Проверка кда на исполняемость|Перебор таблицы разделов|Поиск активного раздела|Чтение первого раздела сектора|

#### 3) GRUB2

Это маленькая умная программа, способная читать файловую систему. Проверив boot, она находит два файла (про второй позднее): vmlinuz - само ядро сжатое в бинарный файл. GRUB распаковывает этот файл в оперативную память и передаёт ему управление.

#### 4) Initrafms

Проблема в том, что ядро загрузилось в память и теперь его задача смонтировать наш основной диск. Но для этого нужны драйвера, которые лежат на диске, для чтения файлов.

Как раз таки для этого в boot есть второй файл - initrafms, который разворачивается в оперативной памяти как виртуальный диск. Внутри него есть мини-версия линукса с минимальнвм набором драйверов. В итоге ядро монитрует этот файл как корень и уже оттуда загружает драйверы для нашего реального железа. После чего выкидывает временное ядро из памяти и заменяет его настоящим.

 #### 5) Kernel

 Мы попали в кольцо защиты - ring0, здесь нам разрешено всё.


### Задачи ядра
#### Управление памятью
Существует механизм виртуальной памяти, позволяющий говорить, что некий виртуальный адрес для некоторого процесса на самом деле хранится в некоторой физической ячейке. Благодаря этому происходит изоляция, так как процессы просто физически не могут залезть в память друг к другу. Также из плюсов можно выделить своппинг - незаметная выгрузка данных с заполненной оперативки в диск. Если же память забита, то появляется oom-killer, который выбирает процессы с большим расходом и низким приоритетом, и закрывает их.

#### Делить время между процессами

Используется вытисняемая многозадачность, где после каждого кванта ядро останавливает выполнение.

#### Абстракция оборудования
Универсальный интерфейс и доступ ПО к железу